In [ ]:
import serial
import numpy as np
import time
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

plt.style.use('dark_background')


In [ ]:
with serial.Serial("COM6", 115200, rtscts=False) as ser:
    ser.write(b'hello\n')
    l = ser.readline().decode()
    print(l)


In [ ]:
def wait_for_control(ser, n, timeout = .1):
    start_timer = time.time()
    while (time.time() - start_timer < timeout):
        if ser.in_waiting > 0:
            b = ser.read()
            if b[0] < 127:
                print(b.decode(), end = "")
                if n < 256:
                    if b == bytes([n]):
                        return 1
            else:
                print(int.from_bytes(b))
                if n < 256:
                    if b == bytes([n]):
                        return 1
    return -1

def data_length_capture(ser, timeout = 1):
    start_timer = time.time()
    while (time.time() - start_timer < timeout):
        if ser.in_waiting > 0:
            b = ser.read()
            if b[0] < 127:
                print(b.decode(), end = "")
            else:
                if b == bytes([150]):
                    t = int.from_bytes(ser.read())
                    l1 = int.from_bytes(ser.read())
                    l2 = int.from_bytes(ser.read())
                    ser.read()
                    ser.read()
                    return l1 * 256 + l2
    return 0


def data_capture(ser, len, timeout = 2):
    start_timer = time.time()
    counter = 0
    while (time.time() - start_timer < timeout):
        if ser.in_waiting > 0:
            b = ser.read()
            if b == bytes([151]):
                b = ser.read(3)                
                break
            else:
                if b[0] < 127:
                    print(b.decode(), end = "")
    
    while (time.time() - start_timer < timeout):
        b = ser.read(4)
        print(int.from_bytes(b, "little"))
        counter += 1            
        if counter == len: break

           
    return 0

In [ ]:
info_outputs = ""
other_outputs = ""
with serial.Serial("COM6", 115200) as ser:
    start_timer = time.time()
    ser.write("S,".encode())

    print(wait_for_control(ser, 212))
    ser.write(bytes([202]))
    data_size = data_length_capture(ser)
    print("data size:", data_size)
    
    ser.write(bytes([200]))
    data_capture(ser, data_size)

    print(wait_for_control(ser, 212))
    ser.write(bytes([202]))
    data_size = data_length_capture(ser)
    print("data size:", data_size)
    
    ser.write(bytes([200]))
    data_capture(ser, data_size)

    print(wait_for_control(ser, 212))
    ser.write(bytes([202]))
    data_size = data_length_capture(ser)
    print("data size:", data_size)
    
    ser.write(bytes([200]))
    data_capture(ser, data_size)



    wait_for_control(ser, 999, 2)

    

In [ ]:
1000*4*8/115200

In [ ]:
info_outputs = ""
other_outputs = ""
with serial.Serial("COM6", 115200) as ser:
    start_timer = time.time()

    ser.write("S,".encode())
    
    while (time.time() - start_timer < 1):
        if ser.in_waiting > 1:
            start_timer = time.time()
            l = ser.readline().decode().strip()
            if l[0] == '[':
                info_outputs += l + "\n"
            elif l[:6] == "Binary":
                while ser.in_waiting > 0:
                    l = ser.read(4)
                    print(int.from_bytes(l, 'little'))

            else:
                other_outputs += l + "\n"
        else:
            time.sleep(.5)
                


In [ ]:
int.from_bytes(l, 'little')

In [ ]:
cmd = '''
set_currents(60, 30, 50);
set_gains(2, 5, 5, 1, 5, 5);
config();


actinic = 100;


set_arr(1,1,0,1,50,50,0,2);

set_arr(1,2,0,1,250,100,actinic,2);
set_arr(2,1,0,1,450,100,actinic,2);
set_arr(3,1,0,1,50,10,actinic,2);
set_arr(3,2,0,1,450,100,0,2);
set_arr(4,1,0,1,50,10,0,2);


run_mpf(0, 0);
run(1,1);
run_mpf(1, actinic);
for(i=0;i<4;i++){
run(2, 1);
run_mpf(1, actinic);
}

run(3,0);
run_mpf(1,0);
for(i=0;i<4;i++){ 
set_arr(4,1,0,1,50+i*10,5,0,2);
run(4,0);
run_mpf(1,0);
}


'''
cmd_str = "C:" + cmd + "?"
cmd_str

In [ ]:
cmd = '''
set_currents(110, 30, 50);
set_gains(1, 5, 5, 1, 5, 5);
config();


actinic = 0;


set_arr(1,1,0,1,50,25,0,2);
set_arr(1,2,0,1,50,25,0,2);

run_mpf(0, 0);
run(1,0);

'''

cmd_str = "C:" + cmd + "?"
cmd_str

In [ ]:
info_outputs = ""
other_outputs = ""
with serial.Serial("COM31", 115200) as ser:
    start_timer = time.time()
    #ser.write("mpf, 0, 100, 100".encode())
    #ser.write("arrun, 500, 500, 0, 0, 0".encode())
    ser.write(cmd_str.encode())
    
    data = {}

    while (time.time() - start_timer < 3000):
        if ser.in_waiting > 2:
            start_timer = time.time()
            l = ser.readline().decode().strip()
            if l[0] == '[':
                info_outputs += l + "\n"


            elif l[:5] == 'Data:':
                _datainfo = l.split("\t")[0]
                _datatype, _datalength = _datainfo.split(",")
                datatype = _datatype.split(":")[1]
                datalength = int(_datalength.split(":")[1])

                if datalength > 0:
                    _data = l.split("\t")[1]
                    if datatype in data:
                        data[datatype] += np.fromstring(_data, sep = ',', dtype=int).tolist()
                    else:
                        data.update({datatype: np.fromstring(_data, sep = ',', dtype=int).tolist()})
               

            elif l == "Cmd Done!":
                break
            else:
                other_outputs += l + "\n"
        else:
            time.sleep(.5)
                


In [ ]:
print(info_outputs, other_outputs)

In [ ]:
x, y = np.array(data['Fluo']), np.array(data['Fluoref'])
plt.plot(x/y)

# if "730" in data:
#     plt.twinx()
#     plt.plot(data['730'] / data['730ref'], 'm')
#     plt.twinx()
#     plt.plot(data['SUN'], 'y')
#     plt.twinx()
#     plt.plot(data['leaf'], 'r')



sig = arr[1:, 1] - 16384
ref = arr[1:, 0] - 16384
plt.plot(ref, sig-ref - ref * 0.01)
plt.plot(ref, sig-ref - ref * 0.01 - 5 * np.sin(ref/3.14/26))
plt.grid(c = [.2, .2, .2, .2])


In [ ]:
y[3000]

In [ ]:
info_outputs = ""
other_outputs = ""
with serial.Serial("COM31", 115200) as ser:
    start_timer = time.time()
    #ser.write("mpf, 0, 100, 100".encode())
    #ser.write("arrun, 500, 500, 0, 0, 0".encode())
    ser.write("sd\n".encode())
    
    data = []

    while (time.time() - start_timer < 40):
        if ser.in_waiting > 2:
            start_timer = time.time()
            l = ser.readline().decode().strip()
            if l[0] == '[':
                info_outputs += l + "\n"


            elif l[:5] == 'Data:':
                data.append(np.fromstring(l[5:], sep = ',', dtype=int))
                
               

            elif l == "Cmd Done!":
                break
            else:
                other_outputs += l + "\n"
        else:
            time.sleep(.5)
                
arr = np.array(data)

In [ ]:
n = 1.006
sig1 = arr[1:, 1] - arr[1:, 0] - (arr[1:, 0] - 16384) * 0.006
ref1= arr[1:, 3] - arr[1:, 2] - (arr[1:, 2] - 16384) * 0.006

sig2 = arr[1:, 5] - arr[1:, 4] - (arr[1:, 4] - 16384 * 2) * 0.003
ref2= arr[1:, 7] - arr[1:, 6] - (arr[1:, 6] - 16384 * 2) * 0.003


sig3 = arr[1:, 9] - arr[1:, 8] - (arr[1:, 8] - 16384 * 3) * 0.0015
ref3= arr[1:, 11] - arr[1:, 10] - (arr[1:, 10] - 16384 * 3) * 0.0015

sig4 = arr[1:, 13] - arr[1:, 12] - (arr[1:, 12] - 16384 * 4) * 0.0008
ref4= arr[1:, 15] - arr[1:, 14] - (arr[1:, 14] - 16384 * 4) * 0.0008

plt.plot(sig1 / ref1 *  1.025 + .1)
plt.plot(sig2 / ref2)



In [ ]:
plt.plot((sig1 * 1) / ref1 , ((sig4 * 1) / ref4) - 1.05* (sig1 * 1) / ref1 - .2)


In [ ]:
(sig2.mean() - 60) / sig1.mean()/2, (sig3.mean() - 120) / sig1.mean()/3, (sig4.mean() - 120) / sig1.mean()/4, 

In [ ]:
x = sig1 / ref1
y = (sig2 - 50) / (ref2)

y2 = (sig4 - 240) / (ref4)
plt.plot(x, y - 1.0 * x)
plt.plot(x, y2 - 1.0 * x)


In [ ]:
np.polyfit(sig1, sig2, deg = 1)

In [ ]:
plt.plot(sig1, sig2)

In [ ]:
(sig2/sig1).mean(), (ref2/ref1).mean()


In [ ]:
tl = []

for x in range(20):
    tl += [x * 500000]
    
_z = tl[-1] + 1500
for x in range(4):
    tl += (np.arange(12) * 120 + x * 1500 + _z).tolist()

_z = tl[-1] + 1500
for x in range(200):
    tl += [x * 1500 + _z]


_z = tl[-1] + 1500
for _ in range(8):
    _z = tl[-1] + 1500
    for x in range(20):
        tl += [x * 1200 + _z]

_z = tl[-1] + 1500
for x in range(50):
    tl += [x * 1500 * 4 + _z]


_z = tl[-1] + 1500
inv = 1000
for x in range(40):
    tl += [x * 1500 * 4 + x * inv + _z]
    inv += 5000

In [ ]:
plt.plot(tl[:], plot_arr[:, 0] / plot_arr[:, 1], '-')
#plt.xscale('log')

In [ ]:
plt.plot((np.array(tl[-50:]) - tl[-50])/1e6, plot_arr[-50:, 0] / plot_arr[-50:, 1], '.-')
#plt.xscale('log')

In [ ]:
print(outputs)

In [ ]:

def line_parse(l:str)->str:
    l = l.strip()
    if len(l) < 1:return ""
    if l[0] == '[':
        print(l)
        return ""
    else:
        return l.strip()

with serial.Serial("COM4", 115200, timeout = 5) as ser:


    ser.write(cmd_str.encode())
    data_ready = False
    checked = False
    ts = time.time()
    while time.time() - ts < 1:
        if ser.in_waiting > 0:
            l = line_parse(ser.readline().decode())
            if l == "Wake!":
                ser.write("Ready".encode())
                data_ready = True
            elif data_ready and l.isdigit():
                ts = time.time()
                data_len = int(l)
                ser.write("GO".encode())
                a1 = ser.read(4 * (data_len + 1))
                _a = np.frombuffer(a1, dtype=">i4")
                if (_a[:-1].sum() == _a[-1]):
                    checked = True
            elif l == "DONE":
                ser.write("Check".encode())
                data_ready = False
                print(_a)

            elif l == "Finish":
                print(l)
                break
            else:
                print(l)

                

In [ ]:
_a[:-1].sum()

In [ ]:
_a

In [ ]:
l